In [ ]:
!gsutil --version

In [ ]:
!gsutil ls gs://sen1floods11/v1.1/data/flood_events/HandLabeled/

In [ ]:
!gsutil du -sh gs://sen1floods11/v1.1/data/flood_events/HandLabeled/S1Hand/
!gsutil du -sh gs://sen1floods11/v1.1/data/flood_events/HandLabeled/LabelHand/
!gsutil du -sh gs://sen1floods11/v1.1/data/flood_events/HandLabeled/S2Hand/
!gsutil du -sh gs://sen1floods11/v1.1/data/flood_events/HandLabeled/JRCWaterHand/
!gsutil du -sh gs://sen1floods11/v1.1/data/flood_events/HandLabeled/S1OtsuLabelHand/

In [ ]:
!mkdir -p /kaggle/working/GeoSentinel_AI

!gsutil -m cp -r \
gs://sen1floods11/v1.1/data/flood_events/HandLabeled/S1Hand \
/kaggle/working/GeoSentinel_AI/

!gsutil -m cp -r \
gs://sen1floods11/v1.1/data/flood_events/HandLabeled/LabelHand \
/kaggle/working/GeoSentinel_AI/

In [ ]:
import os

base = "/kaggle/working/GeoSentinel_AI"

for folder in ["S1Hand", "LabelHand"]:
    path = os.path.join(base, folder)
    files = os.listdir(path) if os.path.exists(path) else []

    print(f"{folder}: {len(files)} files")
    print("Sample:", files[:5])

In [ ]:
import os

base = "/kaggle/working/GeoSentinel_AI"

s1_path = os.path.join(base, "S1Hand")
label_path = os.path.join(base, "LabelHand")

# Remove suffixes to compare common IDs
s1_ids = {
    f.replace("_S1Hand.tif", "")
    for f in os.listdir(s1_path)
    if f.endswith("_S1Hand.tif")
}

label_ids = {
    f.replace("_LabelHand.tif", "")
    for f in os.listdir(label_path)
    if f.endswith("_LabelHand.tif")
}

matched = s1_ids.intersection(label_ids)
missing_labels = s1_ids - label_ids
missing_images = label_ids - s1_ids

print("Total Satellite Images:", len(s1_ids))
print("Total Flood Masks:", len(label_ids))
print("Successfully Matched:", len(matched))
print("Images without Masks:", len(missing_labels))
print("Masks without Images:", len(missing_images))

print("\nSample Matched IDs:")
print(list(sorted(matched))[:10])

In [ ]:
import os
import numpy as np
import rasterio

base = "/kaggle/working/GeoSentinel_AI"

s1_path = os.path.join(base, "S1Hand")
label_path = os.path.join(base, "LabelHand")

# Select one matching image-mask pair
sample_id = sorted(s1_ids.intersection(label_ids))[0]

image_file = os.path.join(s1_path, sample_id + "_S1Hand.tif")
mask_file = os.path.join(label_path, sample_id + "_LabelHand.tif")

# Read satellite image
with rasterio.open(image_file) as src:
    image = src.read()
    print("SATELLITE IMAGE")
    print("Shape:", image.shape)
    print("Bands:", src.count)
    print("Data type:", image.dtype)
    print("CRS:", src.crs)

# Read flood mask
with rasterio.open(mask_file) as src:
    mask = src.read(1)
    print("\nFLOOD MASK")
    print("Shape:", mask.shape)
    print("Data type:", mask.dtype)
    print("Unique pixel values:", np.unique(mask, return_counts=True))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Display VV and VH bands
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(image[0], cmap="gray")
axes[0].set_title("Sentinel-1 VV")

axes[1].imshow(image[1], cmap="gray")
axes[1].set_title("Sentinel-1 VH")

# Mask: show only labelled pixels
display_mask = np.ma.masked_where(mask == -1, mask)

axes[2].imshow(display_mask, cmap="Blues", vmin=0, vmax=1)
axes[2].set_title("Flood Mask (0=Non-flood, 1=Flood)")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
!gsutil ls gs://sen1floods11/v1.1/splits/flood_handlabeled/

In [ ]:
import os

split_path = "/kaggle/working/GeoSentinel_AI/splits"
os.makedirs(split_path, exist_ok=True)

In [ ]:
!gsutil -m cp gs://sen1floods11/v1.1/splits/flood_handlabeled/*.csv /kaggle/working/GeoSentinel_AI/splits/

In [ ]:
import pandas as pd
import os

split_path = "/kaggle/working/GeoSentinel_AI/splits"

for file in os.listdir(split_path):
    if file.endswith(".csv"):
        df = pd.read_csv(os.path.join(split_path, file))

        print("\nFILE:", file)
        print("Shape:", df.shape)
        print("Columns:", df.columns.tolist())
        print(df.head())

In [ ]:
import pandas as pd
import os

split_path = "/kaggle/working/GeoSentinel_AI/splits"

split_files = {
    "train": "flood_train_data.csv",
    "valid": "flood_valid_data.csv",
    "test": "flood_test_data.csv",
    "bolivia": "flood_bolivia_data.csv"
}

split_ids = {}

for split_name, filename in split_files.items():

    df = pd.read_csv(
        os.path.join(split_path, filename),
        header=None
    )

    image_ids = set(df.iloc[:, 0].dropna())

    split_ids[split_name] = image_ids

    print(split_name.upper(), "Images:", len(image_ids))


print("\n--- OVERLAP CHECK ---")

print("Train-Valid:",
      len(split_ids["train"] & split_ids["valid"]))

print("Train-Test:",
      len(split_ids["train"] & split_ids["test"]))

print("Valid-Test:",
      len(split_ids["valid"] & split_ids["test"]))

print("Bolivia-Test:",
      len(split_ids["bolivia"] & split_ids["test"]))

In [ ]:
import os

s1_path = "/kaggle/working/GeoSentinel_AI/S1Hand"
label_path = "/kaggle/working/GeoSentinel_AI/LabelHand"

all_s1 = set(os.listdir(s1_path))
all_labels = set(os.listdir(label_path))

for split_name, ids in split_ids.items():

    missing_images = []
    missing_masks = []

    for filename in ids:

        mask_name = filename.replace(
            "_S1Hand.tif",
            "_LabelHand.tif"
        )

        if filename not in all_s1:
            missing_images.append(filename)

        if mask_name not in all_labels:
            missing_masks.append(mask_name)

    print(f"\n{split_name.upper()}")
    print("Missing Images:", len(missing_images))
    print("Missing Masks:", len(missing_masks))

In [ ]:
import rasterio
import numpy as np
import os
from collections import Counter

mask_counts = Counter()

for filename in os.listdir(label_path):

    if filename.endswith("_LabelHand.tif"):

        mask_file = os.path.join(label_path, filename)

        with rasterio.open(mask_file) as src:
            mask = src.read(1)

        values, counts = np.unique(
            mask,
            return_counts=True
        )

        for value, count in zip(values, counts):
            mask_counts[int(value)] += int(count)

print("TOTAL PIXEL DISTRIBUTION")

for value, count in sorted(mask_counts.items()):
    print(f"Class {value}: {count:,}")

valid_pixels = mask_counts[0] + mask_counts[1]

print("\nVALID PIXELS:", f"{valid_pixels:,}")

print("Non-Flood:", round(mask_counts[0] / valid_pixels * 100, 2), "%")
print("Flood:", round(mask_counts[1] / valid_pixels * 100, 2), "%")
print("Unlabelled:", f"{mask_counts[-1]:,}")

In [ ]:
import rasterio
import numpy as np
import os

sample_file = os.path.join(
    s1_path,
    sorted(os.listdir(s1_path))[0]
)

with rasterio.open(sample_file) as src:
    image = src.read().astype(np.float32)

print("Image Shape:", image.shape)

for i, band in enumerate(image):
    print(f"\nBand {i+1}")

    print("Min:", np.min(band))
    print("Max:", np.max(band))
    print("Mean:", np.mean(band))
    print("Std:", np.std(band))
    print("NaN:", np.isnan(band).sum())
    print("Inf:", np.isinf(band).sum())

In [ ]:
import rasterio
import numpy as np
import os

nan_stats = []

for filename in os.listdir(s1_path):

    if filename.endswith("_S1Hand.tif"):

        filepath = os.path.join(s1_path, filename)

        with rasterio.open(filepath) as src:
            image = src.read().astype(np.float32)

        for band in range(2):

            total = image[band].size
            nan_count = np.isnan(image[band]).sum()
            valid = image[band][np.isfinite(image[band])]

            nan_stats.append({
                "file": filename,
                "band": band + 1,
                "nan_percent": nan_count / total * 100,
                "valid_min": np.min(valid) if len(valid) else np.nan,
                "valid_max": np.max(valid) if len(valid) else np.nan
            })

df_nan = pd.DataFrame(nan_stats)

print("Average NaN % per band:")
print(df_nan.groupby("band")["nan_percent"].mean())

print("\nMaximum NaN %:")
print(df_nan.groupby("band")["nan_percent"].max())

print("\nImages with more than 50% NaN:")
print(
    df_nan[df_nan["nan_percent"] > 50][
        ["file", "band", "nan_percent"]
    ].head(10)
)

In [ ]:
import os
import numpy as np
import rasterio
import pandas as pd

quality_data = []

for filename in os.listdir(s1_path):

    if filename.endswith("_S1Hand.tif"):

        filepath = os.path.join(s1_path, filename)

        with rasterio.open(filepath) as src:
            image = src.read()

        nan_percent = (
            np.isnan(image).sum() / image.size
        ) * 100

        quality_data.append({
            "filename": filename,
            "nan_percent": nan_percent
        })

quality_df = pd.DataFrame(quality_data)

print("Total Images:", len(quality_df))

print("Completely Valid:",
      (quality_df.nan_percent == 0).sum())

print("Partially Valid:",
      ((quality_df.nan_percent > 0) &
       (quality_df.nan_percent < 100)).sum())

print("Completely Invalid:",
      (quality_df.nan_percent == 100).sum())

print("\nNaN Distribution:")

print(
    quality_df["nan_percent"].describe()
)

In [ ]:
invalid_images = quality_df[
    quality_df["nan_percent"] == 100
]

print("Completely Invalid Image:")
print(invalid_images)

In [ ]:
import rasterio
import numpy as np
import os

invalid_mask = "Paraguay_34417_LabelHand.tif"

mask_file = os.path.join(label_path, invalid_mask)

with rasterio.open(mask_file) as src:
    mask = src.read(1)

values, counts = np.unique(mask, return_counts=True)

print("Mask Shape:", mask.shape)
print("\nMask Distribution:")

for value, count in zip(values, counts):
    print(f"Class {value}: {count:,}")

print("\nTotal Pixels:", mask.size)

In [ ]:
invalid_filename = "Paraguay_34417_S1Hand.tif"

clean_split_ids = {}

for split_name, ids in split_ids.items():

    clean_ids = ids - {invalid_filename}

    clean_split_ids[split_name] = clean_ids

    print(
        split_name.upper(),
        "Original:", len(ids),
        "| Clean:", len(clean_ids),
        "| Excluded:", len(ids) - len(clean_ids)
    )

In [ ]:
import os
import numpy as np
import rasterio

band_values = [[], []]

for filename in clean_split_ids["train"]:

    filepath = os.path.join(s1_path, filename)

    with rasterio.open(filepath) as src:
        image = src.read().astype(np.float32)

    for b in range(2):

        valid = image[b][np.isfinite(image[b])]

        if len(valid) > 0:
            band_values[b].append(valid)

for b in range(2):

    values = np.concatenate(band_values[b])

    print(f"\nBand {b+1}")

    print("Valid Pixels:", len(values))
    print("Min:", np.min(values))
    print("Max:", np.max(values))
    print("Mean:", np.mean(values))
    print("Std:", np.std(values))
    print("1st Percentile:", np.percentile(values, 1))
    print("99th Percentile:", np.percentile(values, 99))

In [ ]:
import json
import os

normalization_params = {
    "VV": {
        "mean": -10.392882,
        "std": 4.038803
    },
    "VH": {
        "mean": -17.241083,
        "std": 4.753963
    }
}

params_path = "/kaggle/working/GeoSentinel_AI/normalization.json"

with open(params_path, "w") as f:
    json.dump(normalization_params, f, indent=4)

print("Normalization parameters saved!")
print(normalization_params)

In [ ]:
import torch

print("PyTorch Version:", torch.__version__)

print("CUDA Available:", torch.cuda.is_available())

if torch.cuda.is_available():

    print("GPU Name:", torch.cuda.get_device_name(0))

    print("GPU Count:", torch.cuda.device_count())

    print("GPU Memory:",
          round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
          "GB")

else:
    print("GPU not available")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader

import numpy as np
import rasterio
import os
import json

print("PyTorch Dataset libraries imported successfully!")

In [ ]:
class GeoSentinelDataset(Dataset):

    def __init__(self, image_ids, s1_path, label_path, params):

        self.image_ids = sorted(list(image_ids))
        self.s1_path = s1_path
        self.label_path = label_path
        self.params = params

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):

        image_name = self.image_ids[idx]

        mask_name = image_name.replace(
            "_S1Hand.tif",
            "_LabelHand.tif"
        )

        # Load satellite image
        image_file = os.path.join(
            self.s1_path, image_name
        )

        with rasterio.open(image_file) as src:
            image = src.read().astype(np.float32)

        # Validity mask: both bands must be finite
        valid_pixels = np.isfinite(image).all(axis=0)

        # Normalize VV and VH
        for band, key in enumerate(["VV", "VH"]):

            mean = self.params[key]["mean"]
            std = self.params[key]["std"]

            image[band] = (
                image[band] - mean
            ) / std

        # Replace invalid pixels after normalization
        image = np.nan_to_num(
            image,
            nan=0.0,
            posinf=0.0,
            neginf=0.0
        )

        # Load flood mask
        mask_file = os.path.join(
            self.label_path, mask_name
        )

        with rasterio.open(mask_file) as src:
            mask = src.read(1).astype(np.int64)

        # Valid training pixels
        valid_pixels = valid_pixels & (mask != -1)

        # Convert unlabelled pixels to ignore index
        mask[mask == -1] = 255

        image_tensor = torch.from_numpy(
            image.copy()
        ).float()

        mask_tensor = torch.from_numpy(
            mask.copy()
        ).long()

        valid_tensor = torch.from_numpy(
            valid_pixels.copy()
        ).bool()

        return image_tensor, mask_tensor, valid_tensor

In [ ]:
import os

print("WORKING DIRECTORY:")
print(os.listdir("/kaggle/working"))

print("\nDATASET FOLDER:")
base_path = "/kaggle/working/GeoSentinel_AI"

if os.path.exists(base_path):
    print(os.listdir(base_path))
else:
    print("GeoSentinel_AI folder not found")

print("\nSPLITS FOLDER:")
split_path = os.path.join(base_path, "splits")

if os.path.exists(split_path):
    print(os.listdir(split_path))
else:
    print("Splits folder not found")

In [ ]:
import os

print("KAGGLE INPUT CONTENTS:")

if os.path.exists("/kaggle/input"):
    print(os.listdir("/kaggle/input"))
else:
    print("Input folder not found")

In [ ]:
import os

base_path = "/kaggle/working/GeoSentinel_AI"

s1_path = os.path.join(base_path, "S1Hand")
label_path = os.path.join(base_path, "LabelHand")
split_path = os.path.join(base_path, "splits")

os.makedirs(s1_path, exist_ok=True)
os.makedirs(label_path, exist_ok=True)
os.makedirs(split_path, exist_ok=True)

print("GeoSentinel AI folders created!")

In [ ]:
!gsutil -m cp gs://sen1floods11/v1.1/data/flood_events/HandLabeled/S1Hand/*.tif /kaggle/working/GeoSentinel_AI/S1Hand/

In [ ]:
!gsutil -m cp gs://sen1floods11/v1.1/data/flood_events/HandLabeled/LabelHand/*.tif /kaggle/working/GeoSentinel_AI/LabelHand/

In [ ]:
!gsutil -m cp gs://sen1floods11/v1.1/splits/flood_handlabeled/*.csv /kaggle/working/GeoSentinel_AI/splits/

In [ ]:
import os
import pandas as pd

print("Downloaded Split Files:")

for filename in sorted(os.listdir(split_path)):

    if filename.endswith(".csv"):

        df = pd.read_csv(
            os.path.join(split_path, filename),
            header=None
        )

        print(filename, ":", len(df), "images")

In [ ]:
!gsutil du -sh gs://sen1floods11/v1.1/data/flood_events/HandLabeled/S1Hand/

!gsutil du -sh gs://sen1floods11/v1.1/data/flood_events/HandLabeled/LabelHand/

In [ ]:
!gsutil -m cp gs://sen1floods11/v1.1/data/flood_events/HandLabeled/S1Hand/*.tif /kaggle/working/GeoSentinel_AI/S1Hand/

In [ ]:
!gsutil -m cp gs://sen1floods11/v1.1/data/flood_events/HandLabeled/LabelHand/*.tif /kaggle/working/GeoSentinel_AI/LabelHand/

In [ ]:
import os
import pandas as pd

split_files = {
    "train": "flood_train_data.csv",
    "valid": "flood_valid_data.csv",
    "test": "flood_test_data.csv",
    "bolivia": "flood_bolivia_data.csv"
}

split_ids = {}

for name, filename in split_files.items():

    file_path = os.path.join(split_path, filename)

    df = pd.read_csv(file_path, header=None)

    split_ids[name] = set(df.iloc[:, 0].dropna())

print("Original split counts:")

for name, ids in split_ids.items():
    print(name, len(ids))

In [ ]:
invalid_filename = "Paraguay_34417_S1Hand.tif"

clean_split_ids = {
    name: ids - {invalid_filename}
    for name, ids in split_ids.items()
}

print("Clean split counts:")

for name, ids in clean_split_ids.items():
    print(name, len(ids))

In [ ]:
normalization_params = {
    "VV": {
        "mean": -10.392882,
        "std": 4.038803
    },
    "VH": {
        "mean": -17.241083,
        "std": 4.753963
    }
}

print(normalization_params)

In [ ]:
train_dataset = GeoSentinelDataset(
    clean_split_ids["train"],
    s1_path,
    label_path,
    normalization_params
)

valid_dataset = GeoSentinelDataset(
    clean_split_ids["valid"],
    s1_path,
    label_path,
    normalization_params
)

test_dataset = GeoSentinelDataset(
    clean_split_ids["test"],
    s1_path,
    label_path,
    normalization_params
)

print("Train Dataset:", len(train_dataset))
print("Validation Dataset:", len(valid_dataset))
print("Test Dataset:", len(test_dataset))

In [ ]:
import os

print("Working:", os.listdir("/kaggle/working"))
print("Input:", os.listdir("/kaggle/input"))

In [ ]:
print("Kaggle session active")

In [ ]:
print("Session recovered")

In [ ]:
image, mask, valid = train_dataset[0]

print("Image shape:", image.shape)
print("Mask shape:", mask.shape)
print("Valid mask shape:", valid.shape)

print("\nImage dtype:", image.dtype)
print("Mask dtype:", mask.dtype)

print("\nNaN in image:", torch.isnan(image).sum().item())

print("Mask classes:", torch.unique(mask).tolist())

print("Valid pixels:", valid.sum().item())
print("Ignored pixels:", (~valid).sum().item())

In [ ]:
class GeoSentinelDataset(Dataset):

    def __init__(self, image_ids, s1_path, label_path, params):
        self.image_ids = sorted(list(image_ids))
        self.s1_path = s1_path
        self.label_path = label_path
        self.params = params

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):

        image_name = self.image_ids[idx]

        mask_name = image_name.replace(
            "_S1Hand.tif",
            "_LabelHand.tif"
        )

        with rasterio.open(
            os.path.join(self.s1_path, image_name)
        ) as src:
            image = src.read().astype(np.float32)

        image_valid = np.isfinite(image).all(axis=0)

        for band, key in enumerate(["VV", "VH"]):

            mean = self.params[key]["mean"]
            std = self.params[key]["std"]

            image[band] = (image[band] - mean) / std

        image = np.nan_to_num(
            image,
            nan=0.0,
            posinf=0.0,
            neginf=0.0
        )

        with rasterio.open(
            os.path.join(self.label_path, mask_name)
        ) as src:
            mask = src.read(1).astype(np.int64)

        loss_valid = image_valid & (mask != -1)

        mask[~loss_valid] = 255

        validity_channel = image_valid.astype(
            np.float32
        )[np.newaxis, :, :]

        image = np.concatenate(
            [image, validity_channel],
            axis=0
        )

        image_tensor = torch.from_numpy(
            image.copy()
        ).float()

        mask_tensor = torch.from_numpy(
            mask.copy()
        ).long()

        valid_tensor = torch.from_numpy(
            loss_valid.copy()
        ).bool()

        return image_tensor, mask_tensor, valid_tensor

In [ ]:
train_dataset = GeoSentinelDataset(
    clean_split_ids["train"],
    s1_path,
    label_path,
    normalization_params
)

valid_dataset = GeoSentinelDataset(
    clean_split_ids["valid"],
    s1_path,
    label_path,
    normalization_params
)

test_dataset = GeoSentinelDataset(
    clean_split_ids["test"],
    s1_path,
    label_path,
    normalization_params
)

print("Train:", len(train_dataset))
print("Validation:", len(valid_dataset))
print("Test:", len(test_dataset))

In [ ]:
image, mask, valid = train_dataset[0]

print("Image shape:", image.shape)
print("Mask shape:", mask.shape)
print("Validity shape:", valid.shape)

print("Image NaN:", torch.isnan(image).sum().item())
print("Mask classes:", torch.unique(mask).tolist())
print("Valid pixels:", valid.sum().item())
print("Ignored pixels:", (~valid).sum().item())

In [ ]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("Training batches:", len(train_loader))
print("Validation batches:", len(valid_loader))
print("Testing batches:", len(test_loader))

In [ ]:
images, masks, valid = next(iter(train_loader))

print("Images:", images.shape)
print("Masks:", masks.shape)
print("Validity:", valid.shape)

In [ ]:
import torch
import torch.nn as nn

class DoubleConv(nn.Module):

    def __init__(self, in_channels, out_channels):

        super().__init__()

        self.conv = nn.Sequential(

            nn.Conv2d(
                in_channels, out_channels,
                kernel_size=3, padding=1
            ),

            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),

            nn.Conv2d(
                out_channels, out_channels,
                kernel_size=3, padding=1
            ),

            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.conv(x)


class UNet(nn.Module):

    def __init__(self, in_channels=3, num_classes=2):

        super().__init__()

        self.enc1 = DoubleConv(in_channels, 32)
        self.pool1 = nn.MaxPool2d(2)

        self.enc2 = DoubleConv(32, 64)
        self.pool2 = nn.MaxPool2d(2)

        self.enc3 = DoubleConv(64, 128)
        self.pool3 = nn.MaxPool2d(2)

        self.enc4 = DoubleConv(128, 256)
        self.pool4 = nn.MaxPool2d(2)

        self.bottleneck = DoubleConv(256, 512)

        self.up4 = nn.ConvTranspose2d(512, 256, 2, 2)
        self.dec4 = DoubleConv(512, 256)

        self.up3 = nn.ConvTranspose2d(256, 128, 2, 2)
        self.dec3 = DoubleConv(256, 128)

        self.up2 = nn.ConvTranspose2d(128, 64, 2, 2)
        self.dec2 = DoubleConv(128, 64)

        self.up1 = nn.ConvTranspose2d(64, 32, 2, 2)
        self.dec1 = DoubleConv(64, 32)

        self.output = nn.Conv2d(32, num_classes, 1)

    def forward(self, x):

        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        e3 = self.enc3(self.pool2(e2))
        e4 = self.enc4(self.pool3(e3))

        b = self.bottleneck(self.pool4(e4))

        d4 = self.dec4(
            torch.cat([self.up4(b), e4], dim=1)
        )

        d3 = self.dec3(
            torch.cat([self.up3(d4), e3], dim=1)
        )

        d2 = self.dec2(
            torch.cat([self.up2(d3), e2], dim=1)
        )

        d1 = self.dec1(
            torch.cat([self.up1(d2), e1], dim=1)
        )

        return self.output(d1)

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = UNet(
    in_channels=3,
    num_classes=2
).to(device)

print("Device:", device)
print("Model parameters:",
      sum(p.numel() for p in model.parameters()))

dummy = torch.randn(1, 3, 512, 512).to(device)

with torch.no_grad():
    output = model(dummy)

print("Output shape:", output.shape)

In [ ]:
import numpy as np

class_counts = np.zeros(2, dtype=np.int64)

for i in range(len(train_dataset)):

    _, mask, valid = train_dataset[i]

    valid_labels = mask[valid].numpy()

    counts = np.bincount(
        valid_labels,
        minlength=2
    )

    class_counts += counts[:2]

print("Non-Flood pixels:", class_counts[0])
print("Flood pixels:", class_counts[1])

print(
    "Flood percentage:",
    round(class_counts[1] / class_counts.sum() * 100, 2),
    "%"
)

In [ ]:
class_weights = np.sqrt(
    class_counts.sum() / (2 * class_counts)
)

class_weights = torch.tensor(
    class_weights,
    dtype=torch.float32
).to(device)

print("Class weights:", class_weights)

In [ ]:
print("CUDA available:", torch.cuda.is_available())
print("Device:", device)
print("Training batches:", len(train_loader))
print("Validation batches:", len(valid_loader))

In [ ]:
import torch.nn as nn
import torch.optim as optim

criterion = nn.CrossEntropyLoss(
    weight=class_weights,
    ignore_index=255
)

optimizer = optim.AdamW(
    model.parameters(),
    lr=0.001,
    weight_decay=1e-4
)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=3
)

print("Loss function:", criterion)
print("Optimizer:", optimizer)
print("Learning rate:", optimizer.param_groups[0]["lr"])

In [ ]:
model.train()

images, masks, valid = next(iter(train_loader))

images = images.to(device)
masks = masks.to(device)
valid = valid.to(device)

optimizer.zero_grad()

outputs = model(images)

loss = criterion(outputs, masks)

loss.backward()

print("Input shape:", images.shape)
print("Output shape:", outputs.shape)
print("Loss:", loss.item())

print(
    "Gradient calculated:",
    model.enc1.conv[0].weight.grad is not None
)

optimizer.zero_grad()

In [ ]:
import os
import time
import torch

# Smaller batch for safer GPU training
train_loader = DataLoader(
    train_dataset,
    batch_size=2,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=2,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

# Checkpoint directory
checkpoint_dir = "/kaggle/working/GeoSentinel_AI/checkpoints"
os.makedirs(checkpoint_dir, exist_ok=True)

best_val_loss = float("inf")
epochs = 20

history = {
    "train_loss": [],
    "val_loss": []
}

for epoch in range(epochs):

    start_time = time.time()

    # TRAINING
    model.train()
    train_loss = 0.0

    for images, masks, valid in train_loader:

        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        outputs = model(images)
        loss = criterion(outputs, masks)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    train_loss /= len(train_loader)

    # VALIDATION
    model.eval()
    val_loss = 0.0

    with torch.no_grad():

        for images, masks, valid in valid_loader:

            images = images.to(device, non_blocking=True)
            masks = masks.to(device, non_blocking=True)

            outputs = model(images)
            loss = criterion(outputs, masks)

            val_loss += loss.item()

    val_loss /= len(valid_loader)

    # Learning rate adjustment
    scheduler.step(val_loss)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)

    # Save latest checkpoint every epoch
    torch.save({
        "epoch": epoch + 1,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "best_val_loss": best_val_loss,
        "history": history
    }, os.path.join(checkpoint_dir, "last_checkpoint.pth"))

    # Save best model
    if val_loss < best_val_loss:

        best_val_loss = val_loss

        torch.save(
            model.state_dict(),
            os.path.join(checkpoint_dir, "best_model.pth")
        )

    elapsed = time.time() - start_time

    print(
        f"Epoch [{epoch+1}/{epochs}] "
        f"| Train Loss: {train_loss:.4f} "
        f"| Val Loss: {val_loss:.4f} "
        f"| Best Val: {best_val_loss:.4f} "
        f"| Time: {elapsed/60:.2f} min"
    )

print("\nTraining completed!")
print("Best validation loss:", best_val_loss)

In [ ]:
print("Checking validation images...\n")

for i in range(len(valid_dataset)):

    image, mask, valid = valid_dataset[i]

    valid_count = valid.sum().item()

    if valid_count == 0:
        print("Problem image index:", i)
        print("Mask classes:", torch.unique(mask).tolist())
        print("Valid pixels:", valid_count)

In [ ]:
for i, (images, masks, valid) in enumerate(valid_loader):

    valid_counts = valid.sum(dim=(1, 2))

    if (valid_counts == 0).any():
        print("Problem batch:", i)
        print("Valid counts:", valid_counts.tolist())

In [ ]:
import torch.nn.functional as F

def safe_loss(outputs, masks, valid):

    loss_map = F.cross_entropy(
        outputs,
        masks,
        weight=class_weights,
        ignore_index=255,
        reduction="none"
    )

    valid = valid & (masks != 255)

    if valid.sum() == 0:
        return None

    return loss_map[valid].mean()

In [ ]:
model.eval()

total_loss = 0.0
total_pixels = 0

with torch.no_grad():

    for images, masks, valid in valid_loader:

        images = images.to(device)
        masks = masks.to(device)
        valid = valid.to(device)

        outputs = model(images)

        loss_map = F.cross_entropy(
            outputs,
            masks,
            weight=class_weights,
            ignore_index=255,
            reduction="none"
        )

        valid = valid & (masks != 255)

        if valid.sum() == 0:
            continue

        total_loss += loss_map[valid].sum().item()
        total_pixels += valid.sum().item()

validation_loss = total_loss / total_pixels

print("Validation Loss:", validation_loss)
print("Valid validation pixels:", total_pixels)

In [ ]:
epochs = 20

best_val_loss = float("inf")

history = {
    "train_loss": [],
    "val_loss": []
}

for epoch in range(epochs):

    start_time = time.time()

    # TRAINING
    model.train()

    train_total_loss = 0.0
    train_total_pixels = 0

    for images, masks, valid in train_loader:

        images = images.to(device)
        masks = masks.to(device)
        valid = valid.to(device)

        optimizer.zero_grad(set_to_none=True)

        outputs = model(images)

        loss = safe_loss(outputs, masks, valid)

        if loss is None:
            continue

        loss.backward()
        optimizer.step()

        valid_count = (valid & (masks != 255)).sum().item()

        train_total_loss += loss.item() * valid_count
        train_total_pixels += valid_count

    train_loss = train_total_loss / train_total_pixels

    # VALIDATION
    model.eval()

    val_total_loss = 0.0
    val_total_pixels = 0

    with torch.no_grad():

        for images, masks, valid in valid_loader:

            images = images.to(device)
            masks = masks.to(device)
            valid = valid.to(device)

            outputs = model(images)

            loss = safe_loss(outputs, masks, valid)

            if loss is None:
                continue

            valid_count = (valid & (masks != 255)).sum().item()

            val_total_loss += loss.item() * valid_count
            val_total_pixels += valid_count

    val_loss = val_total_loss / val_total_pixels

    scheduler.step(val_loss)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)

    # SAVE CHECKPOINT
    checkpoint = {
        "epoch": epoch + 1,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "best_val_loss": min(best_val_loss, val_loss),
        "history": history
    }

    torch.save(
        checkpoint,
        os.path.join(checkpoint_dir, "last_checkpoint.pth")
    )

    if val_loss < best_val_loss:

        best_val_loss = val_loss

        torch.save(
            model.state_dict(),
            os.path.join(checkpoint_dir, "best_model.pth")
        )

    elapsed = (time.time() - start_time) / 60

    print(
        f"Epoch [{epoch+1}/{epochs}] "
        f"| Train Loss: {train_loss:.4f} "
        f"| Val Loss: {val_loss:.4f} "
        f"| Best Val: {best_val_loss:.4f} "
        f"| Time: {elapsed:.2f} min"
    )

print("\nTraining completed!")
print("Best validation loss:", best_val_loss)

In [ ]:
best_model_path = os.path.join(
    checkpoint_dir,
    "best_model.pth"
)

model.load_state_dict(
    torch.load(best_model_path, map_location=device)
)

print("Best model loaded successfully!")

In [ ]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=0.0003,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=3
)

best_val_loss = 0.18547417938991112

print("Optimizer initialized!")
print("Learning Rate:", optimizer.param_groups[0]["lr"])
print("Previous Best Val Loss:", best_val_loss)

In [ ]:
print(type(history))
print(history)

In [ ]:
history["train_loss"].append(train_loss)
history["val_loss"].append(val_loss)

print("History updated successfully!")
print("Total training records:", len(history["train_loss"]))
print("Latest Train Loss:", history["train_loss"][-1])
print("Latest Val Loss:", history["val_loss"][-1])

In [ ]:
checkpoint_path = os.path.join(
    checkpoint_dir, "last_checkpoint.pth"
)

torch.save({
    "epoch": 21,
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "best_val_loss": best_val_loss,
    "history": history
}, checkpoint_path)

print("Epoch 21 checkpoint saved successfully!")
print("Best validation loss:", best_val_loss)

In [ ]:
additional_epochs = 60

for epoch in range(22, additional_epochs + 1):

    epoch_start = time.time()

    # -------- TRAINING --------
    model.train()

    train_loss_sum = 0.0
    train_pixels = 0

    for images, masks, valid in train_loader:

        images = images.to(device)
        masks = masks.to(device)
        valid = valid.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = safe_loss(outputs, masks, valid)

        if loss is None:
            continue

        loss.backward()
        optimizer.step()

        pixels = valid.sum().item()
        train_loss_sum += loss.item() * pixels
        train_pixels += pixels

    train_loss = train_loss_sum / train_pixels

    # -------- VALIDATION --------
    model.eval()

    val_loss_sum = 0.0
    val_pixels = 0

    with torch.no_grad():

        for images, masks, valid in valid_loader:

            images = images.to(device)
            masks = masks.to(device)
            valid = valid.to(device)

            outputs = model(images)
            loss = safe_loss(outputs, masks, valid)

            if loss is None:
                continue

            pixels = valid.sum().item()
            val_loss_sum += loss.item() * pixels
            val_pixels += pixels

    val_loss = val_loss_sum / val_pixels

    scheduler.step(val_loss)

    # -------- BEST MODEL --------
    if val_loss < best_val_loss:

        best_val_loss = val_loss

        torch.save(
            model.state_dict(),
            os.path.join(checkpoint_dir, "best_model.pth")
        )

        print("New best model saved!")

    # -------- HISTORY --------
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)

    # -------- CHECKPOINT --------
    torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "best_val_loss": best_val_loss,
        "history": history
    }, checkpoint_path)

    elapsed = (time.time() - epoch_start) / 60

    print(
        f"Epoch [{epoch}/60] | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Best Val: {best_val_loss:.4f} | "
        f"Time: {elapsed:.2f} min"
    )

print("\nTraining completed!")
print("Total epochs:", len(history["train_loss"]))
print("Best Validation Loss:", best_val_loss)

In [ ]:
best_model_path = os.path.join(
    checkpoint_dir, "best_model.pth"
)

model.load_state_dict(
    torch.load(best_model_path, map_location=device)
)

model.eval()

print("Best 60-epoch model loaded successfully!")

In [ ]:
import torch
import numpy as np

# Load best model
best_model_path = os.path.join(
    checkpoint_dir, "best_model.pth"
)

model.load_state_dict(
    torch.load(best_model_path, map_location=device)
)

model.eval()

# Confusion matrix counters
TP = 0
TN = 0
FP = 0
FN = 0

with torch.no_grad():

    for images, masks, valid in test_loader:

        images = images.to(device)
        masks = masks.to(device)
        valid = valid.to(device)

        outputs = model(images)

        predictions = torch.argmax(outputs, dim=1)

        valid_pixels = valid & (masks != 255)

        predictions = predictions[valid_pixels]
        targets = masks[valid_pixels]

        TP += ((predictions == 1) & (targets == 1)).sum().item()
        TN += ((predictions == 0) & (targets == 0)).sum().item()
        FP += ((predictions == 1) & (targets == 0)).sum().item()
        FN += ((predictions == 0) & (targets == 1)).sum().item()


# Calculate metrics
epsilon = 1e-8

precision = TP / (TP + FP + epsilon)
recall = TP / (TP + FN + epsilon)

f1_dice = (2 * TP) / (2 * TP + FP + FN + epsilon)

iou_flood = TP / (TP + FP + FN + epsilon)
iou_background = TN / (TN + FP + FN + epsilon)

mean_iou = (iou_flood + iou_background) / 2

accuracy = (TP + TN) / (TP + TN + FP + FN + epsilon)

specificity = TN / (TN + FP + epsilon)

print("========== GEOSENTINEL AI ==========")
print("FINAL TEST EVALUATION")
print("===================================")

print(f"Accuracy          : {accuracy:.4f}")
print(f"Precision         : {precision:.4f}")
print(f"Recall            : {recall:.4f}")
print(f"Dice Score        : {f1_dice:.4f}")
print(f"Flood IoU         : {iou_flood:.4f}")
print(f"Background IoU    : {iou_background:.4f}")
print(f"Mean IoU          : {mean_iou:.4f}")
print(f"Specificity       : {specificity:.4f}")

print("\n========== CONFUSION MATRIX ==========")
print("True Positive  :", TP)
print("True Negative  :", TN)
print("False Positive :", FP)
print("False Negative :", FN)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch

model.eval()

images, masks, valid = next(iter(test_loader))

images_device = images.to(device)

with torch.no_grad():
    outputs = model(images_device)
    predictions = torch.argmax(outputs, dim=1)

# Show first 3 test images
num_samples = min(3, images.shape[0])

fig, axes = plt.subplots(num_samples, 3, figsize=(15, 5*num_samples))

if num_samples == 1:
    axes = np.expand_dims(axes, axis=0)

for i in range(num_samples):

    # VV band, denormalized
    satellite = (
        images[i, 0].numpy() * normalization_params["VV"]["std"]
        + normalization_params["VV"]["mean"]
    )

    ground_truth = masks[i].numpy()
    predicted = predictions[i].cpu().numpy()

    # Ignore invalid pixels
    ground_truth = np.where(ground_truth == 255, np.nan, ground_truth)
    predicted = np.where(valid[i].numpy(), predicted, np.nan)

    axes[i, 0].imshow(satellite, cmap="gray")
    axes[i, 0].set_title("Sentinel-1 Satellite Image")

    axes[i, 1].imshow(ground_truth, cmap="Blues", vmin=0, vmax=1)
    axes[i, 1].set_title("Ground Truth Flood Mask")

    axes[i, 2].imshow(predicted, cmap="Blues", vmin=0, vmax=1)
    axes[i, 2].set_title("GeoSentinel AI Prediction")

    for ax in axes[i]:
        ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch

model.eval()
selected_samples = []

with torch.no_grad():

    for images, masks, valid in test_loader:

        images_device = images.to(device)

        outputs = model(images_device)
        predictions = torch.argmax(outputs, dim=1)

        for i in range(len(images)):

            gt = masks[i].numpy()
            pred = predictions[i].cpu().numpy()
            valid_mask = valid[i].numpy()

            actual_flood = (gt == 1) & valid_mask
            predicted_flood = (pred == 1) & valid_mask

            flood_pixels = actual_flood.sum()

            # Select images containing sufficient actual flood pixels
            if flood_pixels >= 1000:

                intersection = (actual_flood & predicted_flood).sum()
                union = (actual_flood | predicted_flood).sum()

                sample_iou = intersection / (union + 1e-8)

                satellite = (
                    images[i, 0].numpy()
                    * normalization_params["VV"]["std"]
                    + normalization_params["VV"]["mean"]
                )

                selected_samples.append({
                    "satellite": satellite,
                    "gt": gt.copy(),
                    "pred": pred.copy(),
                    "valid": valid_mask.copy(),
                    "iou": sample_iou,
                    "gt_pixels": int(flood_pixels),
                    "pred_pixels": int(predicted_flood.sum())
                })

            if len(selected_samples) == 4:
                break

        if len(selected_samples) == 4:
            break

print("Selected flood-positive images:", len(selected_samples))

fig, axes = plt.subplots(
    len(selected_samples), 3,
    figsize=(15, 5 * len(selected_samples))
)

if len(selected_samples) == 1:
    axes = np.expand_dims(axes, axis=0)

for i, sample in enumerate(selected_samples):

    gt = np.where(sample["valid"], sample["gt"], np.nan)
    pred = np.where(sample["valid"], sample["pred"], np.nan)

    axes[i, 0].imshow(sample["satellite"], cmap="gray")
    axes[i, 0].set_title("Sentinel-1 Image")

    axes[i, 1].imshow(gt, cmap="Blues", vmin=0, vmax=1)
    axes[i, 1].set_title("Ground Truth")

    axes[i, 2].imshow(pred, cmap="Blues", vmin=0, vmax=1)
    axes[i, 2].set_title(
        f"Prediction | IoU: {sample['iou']:.3f}"
    )

    for ax in axes[i]:
        ax.axis("off")

    print(
        f"Image {i+1}: "
        f"GT Flood Pixels={sample['gt_pixels']}, "
        f"Predicted Flood Pixels={sample['pred_pixels']}, "
        f"IoU={sample['iou']:.4f}"
    )

plt.tight_layout()
plt.show()

In [ ]:
per_image_results = []

model.eval()

with torch.no_grad():

    for images, masks, valid in test_loader:

        images = images.to(device)
        outputs = model(images)
        predictions = torch.argmax(outputs, dim=1)

        for i in range(len(images)):

            gt = masks[i].cpu().numpy()
            pred = predictions[i].cpu().numpy()
            valid_mask = valid[i].cpu().numpy()

            actual_flood = (gt == 1) & valid_mask
            predicted_flood = (pred == 1) & valid_mask

            intersection = np.logical_and(
                actual_flood, predicted_flood
            ).sum()

            union = np.logical_or(
                actual_flood, predicted_flood
            ).sum()

            if actual_flood.sum() > 0:

                image_iou = intersection / (union + 1e-8)

                per_image_results.append(image_iou)

per_image_results = np.array(per_image_results)

print("Images with actual flood:", len(per_image_results))
print("Mean Image IoU:", per_image_results.mean())
print("Median Image IoU:", np.median(per_image_results))

print("\nIoU Distribution:")
print("Below 0.30:", np.sum(per_image_results < 0.30))
print("0.30 - 0.50:", np.sum(
    (per_image_results >= 0.30) & (per_image_results < 0.50)
))
print("0.50 - 0.70:", np.sum(
    (per_image_results >= 0.50) & (per_image_results < 0.70)
))
print("0.70 and above:", np.sum(per_image_results >= 0.70))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.figure(figsize=(9, 5))

plt.hist(
    per_image_results,
    bins=15,
    edgecolor="black"
)

plt.axvline(
    per_image_results.mean(),
    linestyle="--",
    label=f"Mean IoU: {per_image_results.mean():.3f}"
)

plt.axvline(
    np.median(per_image_results),
    linestyle=":",
    label=f"Median IoU: {np.median(per_image_results):.3f}"
)

plt.xlabel("Image-wise Flood IoU")
plt.ylabel("Number of Images")
plt.title("GeoSentinel AI - Test IoU Distribution")
plt.legend()
plt.grid(alpha=0.2)

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch

model.eval()

poor_samples = []
image_index = 0

with torch.no_grad():

    for images, masks, valid in test_loader:

        images_device = images.to(device)
        outputs = model(images_device)
        predictions = torch.argmax(outputs, dim=1)

        for i in range(len(images)):

            gt = masks[i].numpy()
            pred = predictions[i].cpu().numpy()
            valid_mask = valid[i].numpy()

            actual_flood = (gt == 1) & valid_mask
            predicted_flood = (pred == 1) & valid_mask

            if actual_flood.sum() > 0:

                intersection = np.logical_and(
                    actual_flood, predicted_flood
                ).sum()

                union = np.logical_or(
                    actual_flood, predicted_flood
                ).sum()

                iou = intersection / (union + 1e-8)

                satellite = (
                    images[i, 0].numpy()
                    * normalization_params["VV"]["std"]
                    + normalization_params["VV"]["mean"]
                )

                poor_samples.append({
                    "index": image_index,
                    "iou": iou,
                    "satellite": satellite,
                    "gt": gt.copy(),
                    "pred": pred.copy(),
                    "valid": valid_mask.copy()
                })

            image_index += 1

# Sort by lowest IoU
poor_samples.sort(key=lambda x: x["iou"])

print("Four lowest-IoU test images:")

for sample in poor_samples[:4]:
    print(
        f"Image Index: {sample['index']} | "
        f"IoU: {sample['iou']:.4f}"
    )

# Visualize
fig, axes = plt.subplots(4, 3, figsize=(15, 18))

for i, sample in enumerate(poor_samples[:4]):

    gt = np.where(sample["valid"], sample["gt"], np.nan)
    pred = np.where(sample["valid"], sample["pred"], np.nan)

    axes[i, 0].imshow(sample["satellite"], cmap="gray")
    axes[i, 0].set_title("Sentinel-1 Image")

    axes[i, 1].imshow(gt, cmap="Blues", vmin=0, vmax=1)
    axes[i, 1].set_title("Ground Truth")

    axes[i, 2].imshow(pred, cmap="Blues", vmin=0, vmax=1)
    axes[i, 2].set_title(f"Prediction | IoU: {sample['iou']:.3f}")

    for ax in axes[i]:
        ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
print("========== POOR IMAGE ERROR ANALYSIS ==========")

for sample in poor_samples[:4]:

    valid_mask = sample["valid"]

    gt_flood = (
        (sample["gt"] == 1) & valid_mask
    )

    pred_flood = (
        (sample["pred"] == 1) & valid_mask
    )

    intersection = np.logical_and(
        gt_flood, pred_flood
    ).sum()

    print(f"\nImage Index: {sample['index']}")
    print(f"IoU: {sample['iou']:.4f}")
    print(f"Actual Flood Pixels: {gt_flood.sum()}")
    print(f"Predicted Flood Pixels: {pred_flood.sum()}")
    print(f"Correctly Detected Pixels: {intersection}")

    print(
        "Detection Recall:",
        round(
            intersection / (gt_flood.sum() + 1e-8),
            4
        )
    )

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(4, 2, figsize=(12, 18))

for i, sample in enumerate(poor_samples[:4]):

    gt = (sample["gt"] == 1) & sample["valid"]
    pred = (sample["pred"] == 1) & sample["valid"]

    # Confusion map
    error_map = np.zeros((*gt.shape, 3), dtype=np.float32)

    # False Negative = Blue
    error_map[gt & ~pred] = [0, 0, 1]

    # False Positive = Red
    error_map[~gt & pred] = [1, 0, 0]

    # True Positive = Green
    error_map[gt & pred] = [0, 1, 0]

    # Invalid pixels = White
    error_map[~sample["valid"]] = [1, 1, 1]

    axes[i, 0].imshow(sample["gt"], cmap="Blues", vmin=0, vmax=1)
    axes[i, 0].set_title(
        f"Ground Truth | IoU: {sample['iou']:.4f}"
    )

    axes[i, 1].imshow(error_map)
    axes[i, 1].set_title(
        "Error Map: Blue=FN, Red=FP, Green=TP"
    )

    for ax in axes[i]:
        ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(4, 3, figsize=(15, 18))

for i, sample in enumerate(poor_samples[:4]):

    gt = (sample["gt"] == 1) & sample["valid"]
    pred = (sample["pred"] == 1) & sample["valid"]

    # Corrected Ground Truth
    gt_display = np.where(
        sample["valid"], sample["gt"], np.nan
    )

    # Error map
    error_map = np.zeros((*gt.shape, 3), dtype=np.float32)

    error_map[gt & ~pred] = [0, 0, 1]       # FN
    error_map[~gt & pred] = [1, 0, 0]       # FP
    error_map[gt & pred] = [0, 1, 0]        # TP
    error_map[~sample["valid"]] = [1, 1, 1] # Ignore

    axes[i, 0].imshow(sample["satellite"], cmap="gray")
    axes[i, 0].set_title("Sentinel-1 Image")

    axes[i, 1].imshow(
        gt_display, cmap="Blues", vmin=0, vmax=1
    )
    axes[i, 1].set_title("Corrected Ground Truth")

    axes[i, 2].imshow(error_map)
    axes[i, 2].set_title(
        f"Error Map | IoU: {sample['iou']:.4f}"
    )

    for ax in axes[i]:
        ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

IGNORE = 255

def error_codes(gt, pred):
    """0=TN, 1=TP, 2=FP, 3=FN, 4=invalid"""
    valid = gt != IGNORE
    g = (gt == 1) & valid
    p = (pred == 1) & valid
    out = np.zeros(gt.shape, dtype=np.uint8)
    out[g & p] = 1
    out[~g & p & valid] = 2
    out[g & ~p] = 3
    out[~valid] = 4
    return out

def iou_valid(gt, pred):
    codes = error_codes(gt, pred)
    tp, fp, fn = (codes == 1).sum(), (codes == 2).sum(), (codes == 3).sum()
    union = tp + fp + fn
    return np.nan if union == 0 else tp / union

err_cmap = ListedColormap(["black", "green", "red", "blue", "white"])
gt_cmap = ListedColormap(["black", "deepskyblue"]).with_extremes(bad="lightgray")

def show_samples(images, gts, preds, n=4):
    fig, ax = plt.subplots(n, 3, figsize=(10, 3.2 * n))
    for i in range(n):
        img, gt, pred = images[i], gts[i], preds[i]
        lo, hi = np.percentile(img, [2, 98])           # SAR contrast stretch
        ax[i, 0].imshow(img, cmap="gray", vmin=lo, vmax=hi)
        ax[i, 0].set_title("Sentinel-1 Image")

        gt_show = np.ma.masked_where(gt == IGNORE, gt)  # invalid hidden
        ax[i, 1].imshow(gt_show, cmap=gt_cmap, vmin=0, vmax=1, interpolation="nearest")
        ax[i, 1].set_title("Ground Truth (invalid hidden)")

        iou = iou_valid(gt, pred)
        ax[i, 2].imshow(error_codes(gt, pred), cmap=err_cmap, vmin=0, vmax=4, interpolation="nearest")
        ax[i, 2].set_title(f"Error Map | IoU: {'N/A' if np.isnan(iou) else f'{iou:.4f}'}")
        for a in ax[i]: a.axis("off")

    legend = [Patch(color=c, label=l) for c, l in
              [("black","TN"),("green","TP"),("red","FP"),("blue","FN"),("white","Invalid"),("lightgray","Invalid (GT)")]]
    fig.legend(handles=legend, loc="lower center", ncol=6)
    plt.tight_layout(rect=[0, 0.03, 1, 1])
    plt.show()

In [ ]:
import os
import json
import shutil
import torch

# Backup directory
backup_dir = "/kaggle/working/GeoSentinel_AI_Backup"

os.makedirs(backup_dir, exist_ok=True)

# 1. Save best model
shutil.copy2(
    os.path.join(checkpoint_dir, "best_model.pth"),
    os.path.join(backup_dir, "best_model.pth")
)

# 2. Save latest training checkpoint
shutil.copy2(
    os.path.join(checkpoint_dir, "last_checkpoint.pth"),
    os.path.join(backup_dir, "last_checkpoint.pth")
)

# 3. Save project metadata
metadata = {
    "project": "GeoSentinel AI",
    "model": "U-Net",
    "input_channels": 3,
    "output_classes": 2,
    "image_size": 512,
    "total_epochs": 60,
    "best_validation_loss": float(best_val_loss),

    "normalization_params": normalization_params,

    "class_weights": class_weights.detach().cpu().tolist(),

    "test_metrics": {
        "accuracy": 0.9433,
        "precision": 0.7377,
        "recall": 0.8368,
        "dice_score": 0.7841,
        "flood_iou": 0.6449,
        "background_iou": 0.9368,
        "mean_iou": 0.7909,
        "specificity": 0.9583
    },

    "confusion_matrix": {
        "TP": 2104524,
        "TN": 17180797,
        "FP": 748154,
        "FN": 410484
    },

    "training_history": history
}

with open(
    os.path.join(backup_dir, "project_metadata.json"),
    "w"
) as f:
    json.dump(metadata, f, indent=4)

# 4. Create ZIP
zip_path = shutil.make_archive(
    "/kaggle/working/GeoSentinel_AI_Backup",
    "zip",
    backup_dir
)

print("BACKUP CREATED SUCCESSFULLY!")
print("Backup path:", zip_path)
print("Backup size:", round(os.path.getsize(zip_path) / (1024**2), 2), "MB")

In [ ]:
import zipfile

zip_path = "/kaggle/working/GeoSentinel_AI_Backup.zip"

with zipfile.ZipFile(zip_path, "r") as z:
    print("Backup contents:")

    for file in z.namelist():
        print("✓", file)

    print("\nZIP integrity:", z.testzip())

In [ ]:
import torch
import os

print("PyTorch Version:", torch.__version__)
print("GPU Available:", torch.cuda.is_available())
print("CPU Cores:", os.cpu_count())
print("RAM Check: Ready")

In [ ]:
import os

print("Working Directory:")
print(os.listdir("/kaggle/working"))

print("\nInput Directory:")
print(os.listdir("/kaggle/input"))

In [ ]:
import os
print(os.listdir("/kaggle/input"))

In [ ]:
import os

for root, dirs, files in os.walk("/kaggle/input/datasets"):
    print("Folder:", root)
    for file in files:
        print("File:", file)

In [ ]:
import json
import os

base_path = "/kaggle/input/datasets/amanagrawalagrawal/geosentinel-ai-dataset"

with open(os.path.join(base_path, "project_metadata.json"), "r") as f:
    metadata = json.load(f)

print(json.dumps(metadata, indent=4))

In [ ]:
import json
import os

base_path = "/kaggle/input/datasets/amanagrawalagrawal/geosentinel-ai-dataset"

metadata_path = os.path.join(base_path, "project_metadata.json")

with open(metadata_path, "r") as f:
    metadata = json.load(f)

print(json.dumps(metadata, indent=4))

In [ ]:
import torch
import torch.nn as nn

class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.conv(x)


class UNet(nn.Module):
    def __init__(self, in_channels=3, num_classes=2):
        super().__init__()

        self.encoder1 = DoubleConv(in_channels, 32)
        self.pool1 = nn.MaxPool2d(2)

        self.encoder2 = DoubleConv(32, 64)
        self.pool2 = nn.MaxPool2d(2)

        self.encoder3 = DoubleConv(64, 128)
        self.pool3 = nn.MaxPool2d(2)

        self.encoder4 = DoubleConv(128, 256)
        self.pool4 = nn.MaxPool2d(2)

        self.bottleneck = DoubleConv(256, 512)

        self.upconv4 = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.decoder4 = DoubleConv(512, 256)

        self.upconv3 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.decoder3 = DoubleConv(256, 128)

        self.upconv2 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.decoder2 = DoubleConv(128, 64)

        self.upconv1 = nn.ConvTranspose2d(64, 32, 2, stride=2)
        self.decoder1 = DoubleConv(64, 32)

        self.out = nn.Conv2d(32, num_classes, 1)

    def forward(self, x):
        e1 = self.encoder1(x)
        e2 = self.encoder2(self.pool1(e1))
        e3 = self.encoder3(self.pool2(e2))
        e4 = self.encoder4(self.pool3(e3))

        b = self.bottleneck(self.pool4(e4))

        d4 = self.upconv4(b)
        d4 = self.decoder4(torch.cat([d4, e4], dim=1))

        d3 = self.upconv3(d4)
        d3 = self.decoder3(torch.cat([d3, e3], dim=1))

        d2 = self.upconv2(d3)
        d2 = self.decoder2(torch.cat([d2, e2], dim=1))

        d1 = self.upconv1(d2)
        d1 = self.decoder1(torch.cat([d1, e1], dim=1))

        return self.out(d1)


print("U-Net architecture recreated successfully!")

In [ ]:
model_keys = model.state_dict()
saved_keys = state_dict

print("Saved model keys:", len(saved_keys))
print("Current model keys:", len(model_keys))

print("\nFirst 15 Saved Keys:")
print(list(saved_keys.keys())[:15])

print("\nFirst 15 Current Model Keys:")
print(list(model_keys.keys())[:15])

print("\nShape mismatches:")

for key in saved_keys:
    if key in model_keys:
        if saved_keys[key].shape != model_keys[key].shape:
            print(key, saved_keys[key].shape, model_keys[key].shape)

In [ ]:
print("Original Saved Model Architecture Keys:\n")

prefixes = sorted(set(
    key.split(".")[0] for key in state_dict.keys()
))

for name in prefixes:
    print(name)

In [ ]:
for key, value in state_dict.items():
    if key.endswith("weight") and (
        key.startswith("up") or
        key.startswith("dec") or
        key.startswith("output")
    ):
        print(key, tuple(value.shape))

In [ ]:
import torch
import torch.nn as nn

class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.conv(x)


class UNet(nn.Module):
    def __init__(self, in_channels=3, num_classes=2):
        super().__init__()

        self.enc1 = DoubleConv(in_channels, 32)
        self.enc2 = DoubleConv(32, 64)
        self.enc3 = DoubleConv(64, 128)
        self.enc4 = DoubleConv(128, 256)

        self.pool = nn.MaxPool2d(2)

        self.bottleneck = DoubleConv(256, 512)

        self.up4 = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.dec4 = DoubleConv(512, 256)

        self.up3 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.dec3 = DoubleConv(256, 128)

        self.up2 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec2 = DoubleConv(128, 64)

        self.up1 = nn.ConvTranspose2d(64, 32, 2, stride=2)
        self.dec1 = DoubleConv(64, 32)

        self.output = nn.Conv2d(32, num_classes, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))

        b = self.bottleneck(self.pool(e4))

        d4 = self.up4(b)
        d4 = self.dec4(torch.cat([d4, e4], dim=1))

        d3 = self.up3(d4)
        d3 = self.dec3(torch.cat([d3, e3], dim=1))

        d2 = self.up2(d3)
        d2 = self.dec2(torch.cat([d2, e2], dim=1))

        d1 = self.up1(d2)
        d1 = self.dec1(torch.cat([d1, e1], dim=1))

        return self.output(d1)


print("Original U-Net architecture recreated!")

In [ ]:
device = torch.device("cpu")

model = UNet(in_channels=3, num_classes=2).to(device)

model.load_state_dict(state_dict)
model.eval()

total_params = sum(p.numel() for p in model.parameters())

print("Best model restored successfully!")
print("Device:", device)
print("Parameters:", f"{total_params:,}")
print("Model mode: Evaluation")

In [ ]:
import os

print("Kaggle Input Files:\n")

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        print(os.path.join(root, file))

In [ ]:
import os

print("S1Hand exists:", os.path.exists("/kaggle/working/S1Hand"))
print("LabelHand exists:", os.path.exists("/kaggle/working/LabelHand"))
print("Splits exists:", os.path.exists("/kaggle/working/splits"))

In [ ]:
!gsutil ls gs://sen1floods11/v1.1/data/flood_events/HandLabeled/

In [ ]:
!mkdir -p /kaggle/working/splits

!gsutil cp gs://sen1floods11/v1.1/splits/flood_handlabeled/flood_test_data.csv /kaggle/working/splits/

In [ ]:
import os

print(os.listdir("/kaggle/working/splits"))

In [ ]:
import os

print(os.listdir("/kaggle/working/splits"))

In [ ]:
!mkdir -p /kaggle/working/S1Hand

!gsutil -m cp -r \
gs://sen1floods11/v1.1/data/flood_events/HandLabeled/S1Hand/ \
/kaggle/working/

In [ ]:
import os

print("S1Hand files:", len(os.listdir("/kaggle/working/S1Hand")))

In [ ]:
!gsutil -m cp -r \
gs://sen1floods11/v1.1/data/flood_events/HandLabeled/LabelHand/ \
/kaggle/working/

In [ ]:
import os

print("S1Hand:", len(os.listdir("/kaggle/working/S1Hand")))
print("LabelHand:", len(os.listdir("/kaggle/working/LabelHand")))

In [ ]:
import pandas as pd
import os

csv_path = "/kaggle/working/splits/flood_test_data.csv"

test_df = pd.read_csv(csv_path, header=None)

print("Test CSV shape:", test_df.shape)
print("\nFirst 5 rows:")
print(test_df.head())

print("\nS1Hand files:", len(os.listdir("/kaggle/working/S1Hand")))
print("LabelHand files:", len(os.listdir("/kaggle/working/LabelHand")))

In [ ]:
import pandas as pd
import os

csv_path = "/kaggle/working/splits/flood_test_data.csv"

test_df = pd.read_csv(csv_path, header=None)

# Previously identified invalid image
invalid_image = "Paraguay_34417_S1Hand.tif"

test_df = test_df[
    test_df[0] != invalid_image
].reset_index(drop=True)

# Check image-mask pairs
missing_pairs = []

for _, row in test_df.iterrows():
    image_path = os.path.join("/kaggle/working/S1Hand", row[0])
    mask_path = os.path.join("/kaggle/working/LabelHand", row[1])

    if not os.path.exists(image_path) or not os.path.exists(mask_path):
        missing_pairs.append((row[0], row[1]))

print("Valid test images:", len(test_df))
print("Missing pairs:", len(missing_pairs))
print("Invalid image excluded:", invalid_image)

In [ ]:
print("Normalization Parameters:")
print(metadata["normalization_params"])

print("\nModel Parameters:")
print(sum(p.numel() for p in model.parameters()))

print("\nModel Device:")
print(next(model.parameters()).device)

print("\nModel Evaluation Mode:")
print(not model.training)

In [ ]:
import numpy as np
import rasterio
import matplotlib.pyplot as plt
import os
from matplotlib.colors import ListedColormap

# Four previously identified difficult test indices
selected_indices = [51, 54, 56, 1]

base_s1 = "/kaggle/working/S1Hand"
base_label = "/kaggle/working/LabelHand"

normalization = metadata["normalization_params"]

error_cmap = ListedColormap([
    "black",       # True Negative
    "green",       # True Positive
    "red",         # False Positive
    "blue"         # False Negative
])

for idx in selected_indices:

    row = test_df.iloc[idx]
    image_name = row[0]
    mask_name = row[1]

    # Load satellite image
    with rasterio.open(os.path.join(base_s1, image_name)) as src:
        image = src.read().astype(np.float32)

    # Load ground truth
    with rasterio.open(os.path.join(base_label, mask_name)) as src:
        mask = src.read(1)

    # Validity mask
    image_valid = np.isfinite(image).all(axis=0)
    valid = image_valid & (mask != -1)

    # Normalize VV and VH
    image[0] = (
        image[0] - normalization["VV"]["mean"]
    ) / normalization["VV"]["std"]

    image[1] = (
        image[1] - normalization["VH"]["mean"]
    ) / normalization["VH"]["std"]

    image = np.nan_to_num(image, nan=0.0)

    # Third validity channel
    image = np.concatenate([
        image,
        image_valid[None, :, :].astype(np.float32)
    ], axis=0)

    # Model prediction
    input_tensor = torch.from_numpy(image).unsqueeze(0).to(device)

    with torch.no_grad():
        output = model(input_tensor)
        prediction = output.argmax(dim=1).squeeze(0).cpu().numpy()

    # Error map
    gt = mask.copy()
    gt[~valid] = 255

    error_map = np.zeros(mask.shape, dtype=np.uint8)
    error_map[(gt == 1) & (prediction == 1)] = 1
    error_map[(gt == 0) & (prediction == 1)] = 2
    error_map[(gt == 1) & (prediction == 0)] = 3

    error_map[~valid] = 255

    # Metrics
    tp = np.sum((gt == 1) & (prediction == 1))
    fp = np.sum((gt == 0) & (prediction == 1))
    fn = np.sum((gt == 1) & (prediction == 0))

    iou = tp / (tp + fp + fn) if (tp + fp + fn) else 0

    # Visualization
    fig, axes = plt.subplots(1, 4, figsize=(20, 5))

    axes[0].imshow(image[0], cmap="gray")
    axes[0].set_title("Satellite VV")

    axes[1].imshow(np.where(valid, gt, np.nan),
                   cmap="Blues", vmin=0, vmax=1)
    axes[1].set_title("Ground Truth")

    axes[2].imshow(np.where(valid, prediction, np.nan),
                   cmap="Blues", vmin=0, vmax=1)
    axes[2].set_title("Prediction")

    axes[3].imshow(
        np.ma.masked_where(error_map == 255, error_map),
        cmap=error_cmap,
        vmin=0,
        vmax=3
    )
    axes[3].set_title("Error Map")

    for ax in axes:
        ax.axis("off")

    fig.suptitle(
        f"Test Index {idx} | Flood IoU: {iou:.4f} | "
        f"TP: {tp} FP: {fp} FN: {fn}"
    )

    plt.tight_layout()
    plt.show()

In [ ]:
for idx in selected_indices:
    row = test_df.iloc[idx]

    print("\nTest Index:", idx)
    print("Image:", row[0])
    print("Mask:", row[1])

In [ ]:
import numpy as np
import rasterio
import os
import torch

for idx in selected_indices:

    row = test_df.iloc[idx]
    image_name = row[0]
    mask_name = row[1]

    with rasterio.open(os.path.join(base_s1, image_name)) as src:
        image = src.read().astype(np.float32)

    with rasterio.open(os.path.join(base_label, mask_name)) as src:
        mask = src.read(1)

    image_valid = np.isfinite(image).all(axis=0)
    valid = image_valid & (mask != -1)

    image[0] = (image[0] - normalization["VV"]["mean"]) / normalization["VV"]["std"]
    image[1] = (image[1] - normalization["VH"]["mean"]) / normalization["VH"]["std"]

    image = np.nan_to_num(image, nan=0.0)
    image = np.concatenate([
        image,
        image_valid[None].astype(np.float32)
    ], axis=0)

    x = torch.from_numpy(image).unsqueeze(0).to(device)

    with torch.no_grad():
        pred = model(x).argmax(dim=1).squeeze().numpy()

    gt = mask

    tp = np.sum((gt == 1) & (pred == 1) & valid)
    fp = np.sum((gt == 0) & (pred == 1) & valid)
    fn = np.sum((gt == 1) & (pred == 0) & valid)

    iou = tp / (tp + fp + fn) if tp + fp + fn else 0
    recall = tp / (tp + fn) if tp + fn else 0

    print(f"\n{image_name}")
    print(f"TP: {tp} | FP: {fp} | FN: {fn}")
    print(f"Flood IoU: {iou:.4f}")
    print(f"Recall: {recall:.4f}")

In [ ]:
import torch
import numpy as np

for idx in selected_indices:

    row = test_df.iloc[idx]
    image_name = row[0]

    with rasterio.open(os.path.join(base_s1, image_name)) as src:
        image = src.read().astype(np.float32)

    with rasterio.open(os.path.join(base_label, row[1])) as src:
        mask = src.read(1)

    image_valid = np.isfinite(image).all(axis=0)
    valid = image_valid & (mask != -1)

    image[0] = (image[0] - normalization["VV"]["mean"]) / normalization["VV"]["std"]
    image[1] = (image[1] - normalization["VH"]["mean"]) / normalization["VH"]["std"]

    image = np.nan_to_num(image, nan=0.0)

    image = np.concatenate([
        image,
        image_valid[None].astype(np.float32)
    ], axis=0)

    x = torch.from_numpy(image).unsqueeze(0).to(device)

    with torch.no_grad():
        probabilities = torch.softmax(model(x), dim=1)

    flood_prob = probabilities[0, 1].numpy()

    actual_flood = mask == 1
    missed = actual_flood & (np.argmax(probabilities[0].numpy(), axis=0) == 0) & valid

    print("\nImage:", image_name)
    print("Mean flood probability (actual flood):",
          flood_prob[actual_flood & valid].mean())
    print("Mean flood probability (missed pixels):",
          flood_prob[missed].mean() if missed.any() else "No missed pixels")
    print("Maximum flood probability:", flood_prob[valid].max())

In [ ]:
for idx in selected_indices:

    row = test_df.iloc[idx]
    image_name = row[0]

    with rasterio.open(os.path.join(base_s1, image_name)) as src:
        image = src.read().astype(np.float32)

    with rasterio.open(os.path.join(base_label, row[1])) as src:
        mask = src.read(1)

    image_valid = np.isfinite(image).all(axis=0)
    valid = image_valid & (mask != -1)

    image[0] = (image[0] - normalization["VV"]["mean"]) / normalization["VV"]["std"]
    image[1] = (image[1] - normalization["VH"]["mean"]) / normalization["VH"]["std"]

    image = np.nan_to_num(image, nan=0.0)

    image = np.concatenate([
        image,
        image_valid[None].astype(np.float32)
    ], axis=0)

    x = torch.from_numpy(image).unsqueeze(0).to(device)

    with torch.no_grad():
        probs = torch.softmax(model(x), dim=1)[0, 1].numpy()

    flood_probs = probs[(mask == 1) & valid]

    print("\nImage:", image_name)
    print("Flood pixels:", len(flood_probs))

    for threshold in [0.3, 0.5, 0.7]:
        percentage = np.mean(flood_probs >= threshold) * 100
        print(f"Flood pixels with probability >= {threshold}: {percentage:.2f}%")

In [ ]:
!gsutil cp \
gs://sen1floods11/v1.1/splits/flood_handlabeled/flood_valid_data.csv \
/kaggle/working/splits/

In [ ]:
import pandas as pd

valid_df = pd.read_csv(
    "/kaggle/working/splits/flood_valid_data.csv",
    header=None
)

print("Validation images:", len(valid_df))
print(valid_df.head())

In [ ]:
from collections import Counter

countries = [
    filename.split("_")[0]
    for filename in valid_df[0]
]

print("Validation images by country:")
print(Counter(countries))

In [ ]:
import os
import rasterio
import numpy as np
from collections import Counter

valid_image_stats = []

for _, row in valid_df.iterrows():
    filename = row[0]
    image_path = os.path.join(base_s1, filename)
    label_path = os.path.join(base_label, filename.replace("S1Hand", "LabelHand"))

    with rasterio.open(image_path) as src:
        image = src.read()

    with rasterio.open(label_path) as src:
        mask = src.read(1)

    valid_pixels = np.isfinite(image).all(axis=0)
    labeled_pixels = (mask != -1) & valid_pixels

    valid_image_stats.append({
        "filename": filename,
        "image_valid": valid_pixels.mean() * 100,
        "labeled_pixels": labeled_pixels.sum(),
        "flood_pixels": ((mask == 1) & labeled_pixels).sum(),
        "non_flood_pixels": ((mask == 0) & labeled_pixels).sum()
    })

print("Validation images checked:", len(valid_image_stats))
print("Images with no valid pixels:",
      sum(x["image_valid"] == 0 for x in valid_image_stats))
print("Images with no labeled pixels:",
      sum(x["labeled_pixels"] == 0 for x in valid_image_stats))
print("Images with flood pixels:",
      sum(x["flood_pixels"] > 0 for x in valid_image_stats))

In [ ]:
for item in valid_image_stats:
    if item["labeled_pixels"] == 0:
        print("Filename:", item["filename"])

        label_path = os.path.join(
            base_label,
            item["filename"].replace("S1Hand", "LabelHand")
        )

        with rasterio.open(label_path) as src:
            mask = src.read(1)

        print("Mask values:", np.unique(mask, return_counts=True))
        print("-" * 50)

In [ ]:
valid_clean_df = valid_df[
    valid_df[0].isin([
        item["filename"]
        for item in valid_image_stats
        if item["labeled_pixels"] > 0
    ])
].reset_index(drop=True)

print("Original validation images:", len(valid_df))
print("Clean validation images:", len(valid_clean_df))
print("Excluded images:", len(valid_df) - len(valid_clean_df))

In [ ]:
country_stats = {}

for item in valid_image_stats:
    country = item["filename"].split("_")[0]

    if country not in country_stats:
        country_stats[country] = {
            "total": 0,
            "usable": 0,
            "flood_images": 0
        }

    country_stats[country]["total"] += 1

    if item["labeled_pixels"] > 0:
        country_stats[country]["usable"] += 1

    if item["flood_pixels"] > 0:
        country_stats[country]["flood_images"] += 1

for country, stats in country_stats.items():
    print(country, stats)

In [ ]:
from torch.utils.data import Dataset, DataLoader
import numpy as np
import torch
import os
import rasterio

class FloodDataset(Dataset):
    def __init__(self, dataframe):
        self.df = dataframe.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        filename = self.df.iloc[idx, 0]

        image_path = os.path.join(base_s1, filename)
        label_path = os.path.join(
            base_label, filename.replace("S1Hand", "LabelHand")
        )

        with rasterio.open(image_path) as src:
            image = src.read().astype(np.float32)

        with rasterio.open(label_path) as src:
            mask = src.read(1).astype(np.int64)

        image_valid = np.isfinite(image).all(axis=0)
        valid = image_valid & (mask != -1)

        image[0] = (image[0] - normalization["vv_mean"]) / normalization["vv_std"]
        image[1] = (image[1] - normalization["vh_mean"]) / normalization["vh_std"]

        image = np.nan_to_num(image, nan=0.0, posinf=0.0, neginf=0.0)

        validity_channel = image_valid.astype(np.float32)[None, :, :]
        image = np.concatenate([image, validity_channel], axis=0)

        mask[~valid] = 255

        return (
            torch.tensor(image, dtype=torch.float32),
            torch.tensor(mask, dtype=torch.long),
            torch.tensor(valid, dtype=torch.bool)
        )

print("FloodDataset restored successfully!")

In [ ]:
print("Normalization dictionary:")
print(normalization)

print("\nMetadata normalization parameters:")
print(metadata["normalization_params"])


In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import rasterio
from collections import defaultdict
from torch.utils.data import Dataset, DataLoader

# ==========================================
# 1. DATASET CLASS
# ==========================================

class FloodDataset(Dataset):
    def __init__(self, dataframe):
        self.df = dataframe.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        filename = self.df.iloc[idx, 0]

        image_path = os.path.join(base_s1, filename)
        label_path = os.path.join(
            base_label,
            filename.replace("S1Hand", "LabelHand")
        )

        with rasterio.open(image_path) as src:
            image = src.read().astype(np.float32)

        with rasterio.open(label_path) as src:
            mask = src.read(1).astype(np.int64)

        image_valid = np.isfinite(image).all(axis=0)
        valid = image_valid & (mask != -1)

        image[0] = (
            image[0] - normalization["VV"]["mean"]
        ) / normalization["VV"]["std"]

        image[1] = (
            image[1] - normalization["VH"]["mean"]
        ) / normalization["VH"]["std"]

        image = np.nan_to_num(
            image, nan=0.0, posinf=0.0, neginf=0.0
        )

        validity_channel = image_valid.astype(np.float32)[None, :, :]
        image = np.concatenate([image, validity_channel], axis=0)

        mask[~valid] = 255

        return (
            torch.tensor(image, dtype=torch.float32),
            torch.tensor(mask, dtype=torch.long),
            torch.tensor(valid, dtype=torch.bool)
        )


# ==========================================
# 2. DATASET QUALITY AND PIXEL DISTRIBUTION
# ==========================================

print("=" * 60)
print("1. DATASET QUALITY ANALYSIS")
print("=" * 60)

country_stats = defaultdict(lambda: {
    "images": 0, "flood_images": 0,
    "flood": 0, "non_flood": 0
})

total_flood = 0
total_non_flood = 0

for item in valid_image_stats:
    if item["labeled_pixels"] == 0:
        continue

    country = item["filename"].split("_")[0]

    country_stats[country]["images"] += 1
    country_stats[country]["flood"] += item["flood_pixels"]
    country_stats[country]["non_flood"] += item["non_flood_pixels"]

    if item["flood_pixels"] > 0:
        country_stats[country]["flood_images"] += 1

    total_flood += item["flood_pixels"]
    total_non_flood += item["non_flood_pixels"]

total_pixels = total_flood + total_non_flood

print("Original images:", len(valid_df))
print("Usable images:", len(valid_clean_df))
print("Excluded images:", len(valid_df) - len(valid_clean_df))
print("Flood pixels:", total_flood)
print("Non-flood pixels:", total_non_flood)
print("Flood percentage:", round(100 * total_flood / total_pixels, 2))
print("Non-flood percentage:", round(100 * total_non_flood / total_pixels, 2))

print("\nCountry-wise dataset:")
for country, stats in sorted(country_stats.items()):
    print(country, dict(stats))


# ==========================================
# 3. MODEL VALIDATION EVALUATION
# ==========================================

print("\n" + "=" * 60)
print("2. MODEL VALIDATION EVALUATION")
print("=" * 60)

valid_dataset = FloodDataset(valid_clean_df)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=0
)

model.eval()

cm = np.zeros((2, 2), dtype=np.int64)
country_cm = defaultdict(lambda: np.zeros((2, 2), dtype=np.int64))
image_results = []

with torch.inference_mode():

    for i, (images, masks, valid) in enumerate(valid_loader):

        images = images.to(device)
        masks = masks.to(device)
        valid = valid.to(device)

        outputs = model(images)
        predictions = torch.argmax(outputs, dim=1)

        valid_mask = valid & (masks != 255)

        y_true = masks[valid_mask].cpu().numpy()
        y_pred = predictions[valid_mask].cpu().numpy()

        tp = int(((y_true == 1) & (y_pred == 1)).sum())
        tn = int(((y_true == 0) & (y_pred == 0)).sum())
        fp = int(((y_true == 0) & (y_pred == 1)).sum())
        fn = int(((y_true == 1) & (y_pred == 0)).sum())

        current_cm = np.array([[tn, fp], [fn, tp]])

        cm += current_cm

        filename = valid_clean_df.iloc[i, 0]
        country = filename.split("_")[0]

        country_cm[country] += current_cm

        image_iou = (
            tp / (tp + fp + fn)
            if tp + fp + fn else 0
        )

        image_results.append({
            "filename": filename,
            "country": country,
            "TP": tp,
            "TN": tn,
            "FP": fp,
            "FN": fn,
            "Flood_IoU": image_iou
        })

        if (i + 1) % 10 == 0:
            print(f"Processed {i + 1}/{len(valid_dataset)} images")

tn, fp = cm[0]
fn, tp = cm[1]

accuracy = (tp + tn) / cm.sum()
precision = tp / (tp + fp) if tp + fp else 0
recall = tp / (tp + fn) if tp + fn else 0
specificity = tn / (tn + fp) if tn + fp else 0
dice = 2 * tp / (2 * tp + fp + fn) if 2 * tp + fp + fn else 0
flood_iou = tp / (tp + fp + fn) if tp + fp + fn else 0
background_iou = tn / (tn + fp + fn) if tn + fp + fn else 0
mean_iou = (flood_iou + background_iou) / 2

print("\nGLOBAL VALIDATION METRICS")
print("Accuracy:", round(accuracy, 4))
print("Precision:", round(precision, 4))
print("Recall:", round(recall, 4))
print("Specificity:", round(specificity, 4))
print("Dice:", round(dice, 4))
print("Flood IoU:", round(flood_iou, 4))
print("Background IoU:", round(background_iou, 4))
print("Mean IoU:", round(mean_iou, 4))

print("\nConfusion Matrix:")
print(cm)


# ==========================================
# 4. COUNTRY-WISE PERFORMANCE
# ==========================================

print("\n" + "=" * 60)
print("3. COUNTRY-WISE PERFORMANCE")
print("=" * 60)

country_rows = []

for country, matrix in sorted(country_cm.items()):

    c_tn, c_fp = matrix[0]
    c_fn, c_tp = matrix[1]

    c_precision = c_tp / (c_tp + c_fp) if c_tp + c_fp else 0
    c_recall = c_tp / (c_tp + c_fn) if c_tp + c_fn else 0

    c_iou = (
        c_tp / (c_tp + c_fp + c_fn)
        if c_tp + c_fp + c_fn else 0
    )

    c_dice = (
        2 * c_tp / (2 * c_tp + c_fp + c_fn)
        if 2 * c_tp + c_fp + c_fn else 0
    )

    country_rows.append({
        "Country": country,
        "Flood_IoU": round(c_iou, 4),
        "Dice": round(c_dice, 4),
        "Precision": round(c_precision, 4),
        "Recall": round(c_recall, 4),
        "TP": int(c_tp),
        "FP": int(c_fp),
        "FN": int(c_fn)
    })

country_df = pd.DataFrame(country_rows)
print(country_df.to_string(index=False))


# ==========================================
# 5. ERROR ANALYSIS
# ==========================================

print("\n" + "=" * 60)
print("4. ERROR ANALYSIS")
print("=" * 60)

results_df = pd.DataFrame(image_results)

print("Images evaluated:", len(results_df))
print("Mean per-image Flood IoU:",
      round(results_df["Flood_IoU"].mean(), 4))
print("Median per-image Flood IoU:",
      round(results_df["Flood_IoU"].median(), 4))

print("\nIoU distribution:")
print("Below 0.30:", (results_df["Flood_IoU"] < 0.30).sum())

print("0.30 - 0.50:", (
    (results_df["Flood_IoU"] >= 0.30) &
    (results_df["Flood_IoU"] < 0.50)
).sum())

print("0.50 - 0.70:", (
    (results_df["Flood_IoU"] >= 0.50) &
    (results_df["Flood_IoU"] < 0.70)
).sum())

print("Above 0.70:", (results_df["Flood_IoU"] >= 0.70).sum())

print("\n10 Lowest-performing images:")
print(
    results_df.sort_values("Flood_IoU")
    .head(10)[["filename", "Flood_IoU", "FP", "FN"]]
    .to_string(index=False)
)

print("\nTotal False Positives:", fp)
print("Total False Negatives:", fn)

print("\n" + "=" * 60)
print("ALL VALIDATION ANALYSIS COMPLETED")
print("=" * 60)

In [ ]:
results_df["IoU_Denominator"] = (
    results_df["TP"] +
    results_df["FP"] +
    results_df["FN"]
)

valid_iou_df = results_df[
    results_df["IoU_Denominator"] > 0
].copy()

print("Images with defined Flood IoU:", len(valid_iou_df))
print("Mean Flood IoU:",
      round(valid_iou_df["Flood_IoU"].mean(), 4))
print("Median Flood IoU:",
      round(valid_iou_df["Flood_IoU"].median(), 4))

print("\n10 genuinely lowest-performing images:")
print(
    valid_iou_df.sort_values("Flood_IoU")[
        ["filename", "Flood_IoU", "FP", "FN"]
    ].head(10).to_string(index=False)
)

In [ ]:
import os

print("GEO SENTINEL PROJECT FILES")
print("=" * 50)

for root, dirs, files in os.walk("/kaggle/working/GeoSentinel_AI"):
    for file in files:
        path = os.path.join(root, file)
        size = os.path.getsize(path) / (1024 * 1024)

        print(f"{os.path.relpath(path, '/kaggle/working/GeoSentinel_AI')} | {size:.2f} MB")

In [ ]:
import os

print("Working directory contents:")
print(os.listdir("/kaggle/working"))

print("\nGeoSentinel_AI folder exists:",
      os.path.exists("/kaggle/working/GeoSentinel_AI"))

In [ ]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        if file.endswith((".pth", ".json", ".ipynb")):
            path = os.path.join(root, file)
            size = os.path.getsize(path) / (1024 * 1024)
            print(f"{path} | {size:.2f} MB")

In [ ]:
import os
import shutil

source = "/kaggle/input/datasets/amanagrawalagrawal/geosentinel-ai-dataset"
destination = "/kaggle/working/GeoSentinel_AI"

os.makedirs(destination, exist_ok=True)

files_to_copy = [
    "best_model.pth",
    "project_metadata.json"
]

for file in files_to_copy:
    shutil.copy2(
        os.path.join(source, file),
        os.path.join(destination, file)
    )
    print("Copied:", file)

print("\nProject folder created successfully!")
print(os.listdir(destination))